In [1]:
import pandas as pd

# -------------------------------
# 1. Warehouses (Supply)
# -------------------------------
warehouses = ['W1', 'W2', 'W3']
capacity = [700, 800, 600]

warehouse_df = pd.DataFrame({
    'Warehouse': warehouses,
    'Capacity': capacity
})

print("Warehouses Data:")
print(warehouse_df)


# -------------------------------
# 2. Markets (Demand)
# -------------------------------
markets = ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7']
demand = [150, 200, 180, 220, 170, 210, 190]

market_df = pd.DataFrame({
    'Market': markets,
    'Demand': demand
})

print("\nMarkets Data:")
print(market_df)


# -------------------------------
# 3. Cost Matrix (Transportation Cost)
# -------------------------------
cost = [
    [4, 6, 8, 5, 7, 6, 5],   # W1 → Markets
    [5, 4, 7, 6, 5, 7, 6],   # W2 → Markets
    [6, 7, 3, 4, 6, 5, 4]    # W3 → Markets
]

cost_df = pd.DataFrame(cost, index=warehouses, columns=markets)

print("\nCost Matrix:")
print(cost_df)


# -------------------------------
# 4. Validation Check
# -------------------------------
total_supply = sum(capacity) #Assumption: Every warehouse starts the planning period fully stocked up to its usable capacity.
total_demand = sum(demand)

print("\nTotal Supply:", total_supply)
print("Total Demand:", total_demand)


Warehouses Data:
  Warehouse  Capacity
0        W1       700
1        W2       800
2        W3       600

Markets Data:
  Market  Demand
0     M1     150
1     M2     200
2     M3     180
3     M4     220
4     M5     170
5     M6     210
6     M7     190

Cost Matrix:
    M1  M2  M3  M4  M5  M6  M7
W1   4   6   8   5   7   6   5
W2   5   4   7   6   5   7   6
W3   6   7   3   4   6   5   4

Total Supply: 2100
Total Demand: 1320


In [2]:
print("Warehouses:", warehouses)
print("Capacity:", capacity)

print("Markets:", markets)
print("Demand:", demand)

print("Cost Matrix:")
for row in cost:
    print(row)

Warehouses: ['W1', 'W2', 'W3']
Capacity: [700, 800, 600]
Markets: ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7']
Demand: [150, 200, 180, 220, 170, 210, 190]
Cost Matrix:
[4, 6, 8, 5, 7, 6, 5]
[5, 4, 7, 6, 5, 7, 6]
[6, 7, 3, 4, 6, 5, 4]


In [3]:
from scipy.optimize import linprog  # linprog is Linear Programming solver.Think of linprog as an optimizer.
import numpy as np

cost_array = np.array(cost)
c = cost_array.flatten()

In [4]:
num_warehouses = len(warehouses)
num_markets = len(markets)

A_supply = []
for i in range(num_warehouses):
    row = [0] * (num_warehouses * num_markets)
    for j in range(num_markets):
        row[i * num_markets + j] = 1
    A_supply.append(row)

b_supply = capacity

In [5]:
A_demand = []
for j in range(num_markets):
    row = [0] * (num_warehouses * num_markets)
    for i in range(num_warehouses):
        row[i * num_markets + j] = -1
    A_demand.append(row)

b_demand = [-d for d in demand]

In [6]:
A = np.vstack([A_supply, A_demand])
b = np.hstack([b_supply, b_demand])

In [7]:
result = linprog(c, A_ub=A, b_ub=b, method='highs')

In [8]:
print("Status:", result.message)
print("Minimum Cost:", result.fun)

Status: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Minimum Cost: 5680.0


In [9]:
shipment = result.x.reshape((num_warehouses, num_markets))

print("\nOptimal Shipment Plan:")
for i in range(num_warehouses):
    for j in range(num_markets):
        print(f"{warehouses[i]} → {markets[j]}: {shipment[i][j]:.2f}")


Optimal Shipment Plan:
W1 → M1: 150.00
W1 → M2: 0.00
W1 → M3: 0.00
W1 → M4: 10.00
W1 → M5: 0.00
W1 → M6: 0.00
W1 → M7: 190.00
W2 → M1: 0.00
W2 → M2: 200.00
W2 → M3: 0.00
W2 → M4: 0.00
W2 → M5: 170.00
W2 → M6: 0.00
W2 → M7: 0.00
W3 → M1: 0.00
W3 → M2: 0.00
W3 → M3: 180.00
W3 → M4: 210.00
W3 → M5: 0.00
W3 → M6: 210.00
W3 → M7: 0.00


In [10]:
import pandas as pd

warehouse_df = pd.DataFrame({
    "Warehouse": warehouses,
    "Capacity": capacity
})

warehouse_df

,Warehouse,Capacity
0,W1,700
1,W2,800
2,W3,600


In [11]:
warehouse_df.to_csv("warehouses.csv", index=False)

In [12]:
market_df = pd.DataFrame({
    "Market": markets,
    "Demand": demand
})

market_df

,Market,Demand
0,M1,150
1,M2,200
2,M3,180
3,M4,220
4,M5,170
5,M6,210
6,M7,190


In [13]:
market_df.to_csv("markets.csv", index=False)

In [14]:
cost_df = pd.DataFrame(cost,
                       index=warehouses,
                       columns=markets)

cost_df

,M1,M2,M3,M4,M5,M6,M7
W1,4,6,8,5,7,6,5
W2,5,4,7,6,5,7,6
W3,6,7,3,4,6,5,4


In [15]:
cost_long = (
    cost_df
    .reset_index()
    .melt(id_vars="index",                #three predefined parameters: id_vars, var_name, value_name
          var_name="Market",
          value_name="Cost_per_Unit")
)

cost_long.rename(columns={"index":"Warehouse"},
                 inplace=True)

cost_long

,Warehouse,Market,Cost_per_Unit
0,W1,M1,4
1,W2,M1,5
2,W3,M1,6
3,W1,M2,6
4,W2,M2,4
5,W3,M2,7
6,W1,M3,8
7,W2,M3,7
8,W3,M3,3
9,W1,M4,5


In [16]:
cost_long.to_csv("transportation_cost.csv", index=False)

In [17]:
shipment_df = pd.DataFrame(
    shipment,
    index=warehouses,
    columns=markets
)

shipment_df

,M1,M2,M3,M4,M5,M6,M7
W1,150.0,0.0,0.0,10.0,0.0,0.0,190.0
W2,0.0,200.0,0.0,0.0,170.0,0.0,0.0
W3,0.0,0.0,180.0,210.0,0.0,210.0,0.0


In [18]:
shipment_long = (
    shipment_df
    .reset_index()
    .melt(id_vars="index",
          var_name="Market",
          value_name="Units_Shipped")
)

shipment_long.rename(columns={"index":"Warehouse"},
                     inplace=True)

shipment_long

,Warehouse,Market,Units_Shipped
0,W1,M1,150.0
1,W2,M1,0.0
2,W3,M1,0.0
3,W1,M2,0.0
4,W2,M2,200.0
5,W3,M2,0.0
6,W1,M3,0.0
7,W2,M3,0.0
8,W3,M3,180.0
9,W1,M4,10.0


In [19]:
shipment_long.to_csv("shipments.csv", index=False)

In [44]:
import os
print(os.getcwd())

C:\Users\nisha


In [46]:
os.listdir()

['.anaconda',
 '.conda',
 '.condarc',
 '.continuum',
 '.cursor',
 '.idlerc',
 '.ipynb_checkpoints',
 '.ipython',
 '.jupyter',
 '.matplotlib',
 '.streamlit',
 '.vscode',
 '100 Generators in Pyhton.ipynb',
 '101 Generators homework.ipynb',
 '104 Collections Module.ipynb',
 '105 OS Modules.ipynb',
 '106 Datetime Module.ipynb',
 '107 Math and random modules.ipynb',
 '108 Python Debugger.ipynb',
 '109 Regular Expressions.ipynb',
 '110 Regex part 2.ipynb',
 '111 Regex 3.ipynb',
 '112 Timing the pyhton code .ipynb',
 '113 Zipping and unzipping.ipynb',
 '116 Intro to Web Scraping.ipynb',
 '118 Grabbing a title .ipynb',
 '119 grab a class.ipynb',
 '119 Grabbing a class.ipynb',
 '120 Grabbing a image.ipynb',
 '121 - 122 Web Scraping.ipynb',
 '124 Web Scraping Exercises.ipynb',
 '126 Working with images.ipynb',
 '129 Intro to pdf and spreadsheet.ipynb',
 '135 Sending emails .ipynb',
 '139 Advanced Numbers.ipynb',
 '140 Advanced sets.ipynb',
 '140 Advanced strings.ipynb',
 '140 Advanced stringsipy